In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.stat import Correlation
from pyspark.sql.window import Window

# 1. Start Fresh with LEGACY time policy
try: spark.stop()
except: pass

spark = SparkSession.builder \
    .appName("US_Accidents_Final_Fixed") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

# 2. Load and Sample
df_raw = spark.read.csv("../../shared/US_Accidents_March23.csv", header=True)
df = df_raw.sample(False, 0.1, seed=42)

# 3. Robust Preprocessing
# We take only the first 19 characters of Start_Time to ignore the nanosecond mess
df = df.withColumn("Start_Time_Clean", F.substring(F.col("Start_Time"), 1, 19))
df = df.withColumn("Start_Time_Parsed", F.to_timestamp("Start_Time_Clean", "yyyy-MM-dd HH:mm:ss"))

# Cast core columns
df = df.withColumn("Severity", F.col("Severity").cast("int")) \
       .withColumn("Temp", F.col("Temperature(F)").cast("float")) \
       .withColumn("Visibility", F.col("Visibility(mi)").cast("float"))

# 4. Replace Nulls with grouped averages
df = df.withColumn("Month", F.month("Start_Time_Parsed"))

w_vis = Window.partitionBy("Weather_Condition")
df = df.withColumn("Visibility", F.coalesce(F.col("Visibility"), F.avg("Visibility").over(w_vis), F.lit(10.0)))

w_temp = Window.partitionBy("State", "Month")
df = df.withColumn("Temp", F.coalesce(F.col("Temp"), F.avg("Temp").over(w_temp), F.lit(65.0)))

df = df.dropna(subset=['Weather_Condition', 'Start_Time_Parsed'])

df = df.dropna()

# 5. Create Time and Road Features
df = df.withColumn("Hour", F.hour("Start_Time_Parsed")) \
       .withColumn("DayOfWeek", F.dayofweek("Start_Time_Parsed")) \
       .withColumn("Junction_Int", F.when(F.col("Junction") == "True", 1).otherwise(0)) \
       .withColumn("Signal_Int", F.when(F.col("Traffic_Signal") == "True", 1).otherwise(0))

# 6. Simplify Weather
df = df.withColumn("Weather_Simple", 
    F.when(F.col("Weather_Condition").contains("Rain"), "Rain")
     .when(F.col("Weather_Condition").contains("Snow"), "Snow")
     .otherwise("Clear"))

# 7. Convert Weather to Numeric
indexer = StringIndexer(inputCol="Weather_Simple", outputCol="Weather_Code")
df_final = indexer.fit(df).transform(df)

# 8. Drop any row that still has a NULL in our target features
feature_cols = ['Temp', 'Visibility', 'Junction_Int', 'Signal_Int', 'Hour', 'DayOfWeek', 'Weather_Code']
df_analysis = df_final.select(['Severity'] + feature_cols).dropna()

print(f"Final Row Count for Correlation: {df_analysis.count()}")

# 9. Run Correlation
if df_analysis.count() > 0:
    assembler = VectorAssembler(inputCols=['Severity'] + feature_cols, outputCol="features")
    vector_df = assembler.transform(df_analysis).select("features")
    
    matrix = Correlation.corr(vector_df, "features").collect()[0][0]
    correlations = matrix.toArray()[0]

    print("\n--- Correlation with Severity ---")
    for i, col in enumerate(['Severity'] + feature_cols):
        print(f"{col:20}: {correlations[i]:.4f}")
else:
    print("Dataset is empty. The date parsing might still be failing.")

Final Row Count for Correlation: 356920


In [6]:
!pip install --user folium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.4/113.4 kB 4.2 MB/s eta 0:00:00


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap

# 1. Sample and Convert to Pandas
# We take 50,000 points - enough to see the 'heat' without crashing your browser
print("Preparing geographic data...")
map_df = df_final.select("Start_Lat", "Start_Lng").dropna().sample(False, 0.05).toPandas()

# 2. Static Density Map (Seaborn)
plt.figure(figsize=(12, 8))
# We use a hexbin plot to show density (hotspots)
plt.hexbin(map_df['Start_Lng'], map_df['Start_Lat'], gridsize=100, cmap='YlOrRd', mincnt=1)
plt.colorbar(label='Accident Density')
plt.title("US Accident Hotspots (Density Map)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.xlim(-125, -66) # Focus on Mainland US
plt.ylim(24, 50)
plt.show()

# 3. Interactive Browser Heatmap (Folium)
# This creates an HTML file you can open to zoom into streets
print("Generating interactive HTML map...")
m = folium.Map(location=[37.0902, -95.7129], zoom_start=4, tiles='cartodbpositron')

# Prepare data for Folium
heat_data = [[row['Start_Lat'], row['Start_Lng']] for index, row in map_df.iterrows()]

# Add Heatmap layer
HeatMap(heat_data, radius=8, blur=5).add_to(m)

# Save and prompt user
m.save("us_accidents_hotspots.html")
print("Interactive map saved as 'us_accidents_hotspots.html'. Open this file in your browser to see street-level hotspots!")

Preparing geographic data...


NameError: name 'df_final' is not defined